In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

In [2]:
warnings.filterwarnings('ignore')

In [3]:
df_device = pd.read_csv("/srv/cert/r4.2/device.csv")

In [4]:
df_device

,id,date,user,pc,activity
0,{J1S3-L9UU75BQ-7790ATPL},01/02/2010 07:21:06,MOH0273,PC-6699,Connect
1,{N7B5-Y7BB27SI-2946PUJK},01/02/2010 07:37:41,MOH0273,PC-6699,Disconnect
2,{U1V9-Z7XT67KV-5649MYHI},01/02/2010 07:59:11,HPH0075,PC-2417,Connect
3,{H0Z7-E6GB57XZ-1603MOXD},01/02/2010 07:59:49,IIW0249,PC-0843,Connect
4,{L7P2-G4PX02RX-7999GYOY},01/02/2010 08:04:26,IIW0249,PC-0843,Disconnect
...,...,...,...,...,...
405375,{R7R7-Y9VH64MN-4427OTOU},05/16/2011 22:27:23,EIS0041,PC-0422,Disconnect
405376,{J1G6-G7KE64TX-7505AXXN},05/16/2011 22:43:49,IBB0359,PC-4176,Connect
405377,{I3V8-Q1KQ57JG-4571IXHJ},05/16/2011 22:48:39,IBB0359,PC-4176,Disconnect
405378,{W9Y8-O7VO98OA-0160JVWR},05/16/2011 23:22:29,IBB0359,PC-3620,Connect


In [ ]:
def extract_device_features_userlevel(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Convertendo para datetime
    df["datetime"] = pd.to_datetime(
        df["date"], dayfirst=False, infer_datetime_format=True, errors="coerce"
    )
    df = df.dropna(subset=["datetime"])

    # Mantendo uma coluna do dia
    df["day"] = df["datetime"].dt.floor("D")

    # Flags de conexão/desconexão
    act = df["activity"].astype(str).str.lower().str.strip()
    df["is_connect"] = (act == "connect").astype(int)
    df["is_disconnect"] = (act == "disconnect").astype(int)

    df["hour_dec"] = (
        df["datetime"].dt.hour
        + df["datetime"].dt.minute / 60
        + df["datetime"].dt.second / 3600
    )

    # Agregando por usuário (user-level)
    features = (
        df.groupby("user", observed=True)
          .agg(
              unique_pcs_used=("pc", "nunique"),
              usb_connects=("is_connect", "sum"),
              usb_disconnects=("is_disconnect", "sum"),
              device_events_total=("activity", "count"),
          )
          .reset_index()
    )

    features["unpaired_connections"] = features["usb_connects"] - features["usb_disconnects"]
    features["unpaired_ratio"] = features["unpaired_connections"] / (features["usb_connects"] + 1e-9)

    # Horários típicos de uso por usuário
    percentiles = (
        df.groupby("user", observed=True)["hour_dec"]
          .agg(
              q05=lambda x: float(np.percentile(x, 5)),
              q95=lambda x: float(np.percentile(x, 95)),
          )
          .reset_index()
    )

    # Marcando eventos fora do horário típico
    df = df.merge(percentiles, on="user", how="left")
    df["is_offhour"] = ((df["hour_dec"] < df["q05"]) | (df["hour_dec"] > df["q95"])).astype(int)

    # Total de eventos fora do horário por usuário
    offhour_user = (
        df.groupby("user", observed=True)["is_offhour"]
          .sum()
          .rename("offhour_device_count")
          .reset_index()
    )
    features = features.merge(offhour_user, on="user", how="left")

    # Razão de eventos fora do horário em relação ao total de eventos de device
    features["offhour_device_ratio"] = features["offhour_device_count"] / (features["device_events_total"] + 1e-9)

    daily_counts = (
        df.groupby(["user", "day"], observed=True)
          .size()
          .rename("device_events_per_day")
          .reset_index()
    )

    #  Estatísticas em nível de usuário
    daily_stats = (
        daily_counts.groupby("user", observed=True)["device_events_per_day"]
                   .agg(
                       mean_daily_device_events="mean",
                       median_daily_device_events="median",
                       max_daily_device_events="max",
                       std_daily_device_events="std",
                       active_days="count",
                   )
                   .reset_index()
    )

    features = features.merge(daily_stats, on="user", how="left")
    features["std_daily_device_events"] = features["std_daily_device_events"].fillna(0.0)

    return features


In [6]:
device_user = extract_device_features_userlevel(df_device)
print(device_user.head())
print("shape:", device_user.shape)

      user  unique_pcs_used  usb_connects  usb_disconnects  \
0  AAF0535                1           346              342   
1  AAM0658                1             7                6   
2  ABC0174                1           642              634   
3  AHD0848                1           150              149   
4  AHM0410                1           394              388   

   device_events_total  unpaired_connections  unpaired_ratio  \
0                  688                     4        0.011561   
1                   13                     1        0.142857   
2                 1276                     8        0.012461   
3                  299                     1        0.006667   
4                  782                     6        0.015228   

   offhour_device_count  offhour_device_ratio  mean_daily_device_events  \
0                    70              0.101744                  4.438710   
1                     2              0.153846                  2.600000   
2                

In [7]:
df_logon = pd.read_csv("/srv/cert/r4.2/logon.csv")

In [ ]:
def extract_logon_features_userlevel(df_logon: pd.DataFrame) -> pd.DataFrame:
    df = df_logon.copy()

    # Converte a coluna de data para datetime
    df["Timestamp_DT"] = pd.to_datetime(
        df["date"], format="%m/%d/%Y %H:%M:%S", errors="coerce"
    )
    df = df.dropna(subset=["Timestamp_DT"])

    # Dia (usado para calcular as features diárias e depois resumir no usuário)
    df["day"] = df["Timestamp_DT"].dt.floor("D")

    # Hora em decimal
    df["hour_dec"] = (
        df["Timestamp_DT"].dt.hour
        + df["Timestamp_DT"].dt.minute / 60.0
        + df["Timestamp_DT"].dt.second / 3600.0
    )

    # Mapeia atividade
    df["activity_id"] = (
        df["activity"].astype(str).str.lower().str.strip()
          .map({"logon": 1, "logoff": 0})
          .astype("Int64")
    )

    # Ordena para operações sequenciais por (user, pc)
    df = df.sort_values(["user", "pc", "Timestamp_DT"])

    # Próxima atividade e timestamp no mesmo (user, pc)
    df["next_act_upc"] = df.groupby(["user", "pc"], observed=True)["activity_id"].shift(-1)
    df["next_time_upc"] = df.groupby(["user", "pc"], observed=True)["Timestamp_DT"].shift(-1)

    # Logon sem logoff logo em seguida
    df["missing_logoff"] = ((df["activity_id"] == 1) & (df["next_act_upc"] != 0)).astype(int)

    # Contador de sessões abertas (logon +1 / logoff -1)
    df["session_inc"] = np.where(df["activity_id"] == 1, 1, -1)
    df["open_sessions"] = (
        df.groupby(["user", "pc"], observed=True)["session_inc"]
          .cumsum()
          .clip(lower=0)
    )

    # Duração da sessão (somente logon->logoff)
    mask_session = (df["activity_id"] == 1) & (df["next_act_upc"] == 0)
    df["session_duration_h"] = np.where(
        mask_session,
        (df["next_time_upc"] - df["Timestamp_DT"]).dt.total_seconds() / 3600.0,
        np.nan
    )

    # Troca de PC ao longo do tempo por usuário
    df = df.sort_values(["user", "Timestamp_DT"])
    prev_pc = df.groupby("user", observed=True)["pc"].shift(1)
    df["pc_switch"] = (df["pc"] != prev_pc).astype(int)

    # Corrige o primeiro evento do usuário 
    first_idx = df.groupby("user", observed=True).head(1).index
    df.loc[first_idx, "pc_switch"] = 0

    # Features diárias 
    g = df.groupby(["user", "day"], observed=True)

    total_logons_day = g["activity_id"].apply(lambda x: int((x == 1).sum())).rename("total_logons_day")
    total_logoffs_day = g["activity_id"].apply(lambda x: int((x == 0).sum())).rename("total_logoffs_day")

    logon_to_logoff_ratio = (
        total_logons_day / total_logoffs_day.replace(0, np.nan)
    ).fillna(total_logons_day).rename("logon_to_logoff_ratio")

    missing_logoff_count = g["missing_logoff"].sum().rename("missing_logoff_count")
    open_sessions_mean = g["open_sessions"].mean().rename("open_sessions_mean")
    open_sessions_max = g["open_sessions"].max().rename("open_sessions_max")
    distinct_pcs_used = g["pc"].nunique().rename("distinct_pcs_used")
    pc_switch_count = g["pc_switch"].sum().rename("pc_switch_count")
    sessions_total_duration = g["session_duration_h"].sum(min_count=1).fillna(0.0).rename("sessions_total_duration")

    df_day = pd.concat([
        total_logons_day,
        total_logoffs_day,
        logon_to_logoff_ratio,
        missing_logoff_count,
        open_sessions_mean,
        open_sessions_max,
        distinct_pcs_used,
        pc_switch_count,
        sessions_total_duration,
    ], axis=1).reset_index()

    # Agora resume para user-level 
    user_features = (
        df_day.groupby("user", observed=True)
              .agg(
                  total_logons_day=("total_logons_day", "sum"),
                  total_logoffs_day=("total_logoffs_day", "sum"),
                  logon_to_logoff_ratio=("logon_to_logoff_ratio", "mean"),
                  missing_logoff_count=("missing_logoff_count", "sum"),
                  open_sessions_mean=("open_sessions_mean", "mean"),
                  open_sessions_max=("open_sessions_max", "max"),
                  distinct_pcs_used=("distinct_pcs_used", "mean"),
                  pc_switch_count=("pc_switch_count", "sum"),
                  sessions_total_duration=("sessions_total_duration", "sum"),
              )
              .reset_index()
    )

    return user_features


In [9]:
logon_user = extract_logon_features_userlevel(df_logon)
print(logon_user.head())
print("shape:", logon_user.shape)

      user  total_logons_day  total_logoffs_day  logon_to_logoff_ratio  \
0  AAE0190               346                346                    1.0   
1  AAF0535               164                164                    1.0   
2  AAF0791               346                346                    1.0   
3  AAL0706               346                346                    1.0   
4  AAM0658               229                229                    1.0   

   missing_logoff_count  open_sessions_mean  open_sessions_max  \
0                     0                 0.5                  1   
1                     0                 0.5                  1   
2                     0                 0.5                  1   
3                     0                 0.5                  1   
4                     0                 0.5                  1   

   distinct_pcs_used  pc_switch_count  sessions_total_duration  
0                1.0                0              3478.333333  
1                1.0        

In [10]:
df_file = pd.read_csv("/srv/cert/r4.2/file.csv")

In [11]:
df_file

,id,date,user,pc,filename,content
0,{L9G8-J9QE34VM-2834VDPB},01/02/2010 07:23:14,MOH0273,PC-6699,EYPC9Y08.doc,D0-CF-11-E0-A1-B1-1A-E1 during difficulty over...
1,{H0W6-L4FG38XG-9897XTEN},01/02/2010 07:26:19,MOH0273,PC-6699,N3LTSU3O.pdf,25-50-44-46-2D carpenters 25 landed strait dis...
2,{M3Z0-O2KK89OX-5716MBIM},01/02/2010 08:12:03,HPH0075,PC-2417,D3D3WC9W.doc,D0-CF-11-E0-A1-B1-1A-E1 union 24 declined impo...
3,{E1I4-S4QS61TG-3652YHKR},01/02/2010 08:17:00,HPH0075,PC-2417,QCSW62YS.doc,D0-CF-11-E0-A1-B1-1A-E1 becoming period begin ...
4,{D4R7-E7JL45UX-0067XALT},01/02/2010 08:24:57,HSB0196,PC-8001,AU75JV6U.jpg,FF-D8
...,...,...,...,...,...,...
445576,{J2Q8-Z4UX02QT-2675RTJI},05/16/2011 23:22:31,IBB0359,PC-3620,R8DTDN2Q.txt,5A-44-4F-39 government operations led wound co...
445577,{S3G4-K5PB37EL-4663ADOI},05/16/2011 23:22:32,IBB0359,PC-3620,BLHCRL6W.pdf,25-50-44-46-2D throughout 20 once armed southe...
445578,{G6G8-X7JK94JA-2703OXCY},05/16/2011 23:22:33,IBB0359,PC-3620,ZHMDTTW0.doc,D0-CF-11-E0-A1-B1-1A-E1 companies river englan...
445579,{H8U9-F5GM78QB-9540DSQM},05/16/2011 23:22:33,IBB0359,PC-3620,AC0QL6KF.pdf,25-50-44-46-2D member august observed selected...


In [ ]:
def build_file_user_from_df(
    df_file: pd.DataFrame,
) -> pd.DataFrame:

    df = df_file.copy()

    needed = {"user", "pc", "filename", "content", "date"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"Faltam colunas no df_file: {missing}")

    df["datetime"] = pd.to_datetime(
        df["date"],
        format="%m/%d/%Y %H:%M:%S",
        errors="coerce"
    )

    acc = {}

    for row in df.itertuples(index=False):
        user = getattr(row, "user", None)
        if user is None or pd.isna(user):
            continue

        if user not in acc:
            acc[user] = {
                "total": 0,
                "pcs": set(),
                "files": set(),
                "contents": set(),
                "exe": 0,
                "zip": 0,
                "pdf": 0,
                "docx": 0,
                "xlsx": 0,
            }

        a = acc[user]
        a["total"] += 1

        pc = getattr(row, "pc", None)
        if pd.notna(pc):
            a["pcs"].add(pc)

        fn = getattr(row, "filename", None)
        if pd.notna(fn):
            a["files"].add(fn)

            fn_l = str(fn).lower()
            if fn_l.endswith(".exe"):
                a["exe"] += 1
            if fn_l.endswith(".zip"):
                a["zip"] += 1
            if fn_l.endswith(".pdf"):
                a["pdf"] += 1
            if fn_l.endswith(".docx"):
                a["docx"] += 1
            if fn_l.endswith(".xlsx"):
                a["xlsx"] += 1

        content = getattr(row, "content", None)
        if pd.notna(content):
            a["contents"].add(content)

    out = []
    for user, a in acc.items():
        total = a["total"]

        out.append({
            "user": user,
            "total_file_events": total,
            "unique_pcs_used": len(a["pcs"]),
            "unique_files": len(a["files"]),
            "unique_file_types": len(a["contents"]),

            "exe_files_accessed": a["exe"],
            "zip_files_accessed": a["zip"],
            "pdf_files_accessed": a["pdf"],
            "docx_files_accessed": a["docx"],
            "xlsx_files_accessed": a["xlsx"],

            "exe_ratio": (a["exe"] / total) if total else 0.0,
            "zip_ratio": (a["zip"] / total) if total else 0.0,
            "pdf_ratio": (a["pdf"] / total) if total else 0.0,
            "docx_ratio": (a["docx"] / total) if total else 0.0,
            "xlsx_ratio": (a["xlsx"] / total) if total else 0.0,
        })

    return pd.DataFrame(out)


In [13]:
file_user = build_file_user_from_df(df_file)
print(file_user.head())
print("shape:", file_user.shape)


      user  total_file_events  unique_pcs_used  unique_files  \
0  MOH0273               9306                1          9306   
1  HPH0075               9323                1          9323   
2  HSB0196              11627                1         11627   
3  RRC0553               1342                1          1342   
4  LRR0148                907                1           907   

   unique_file_types  exe_files_accessed  zip_files_accessed  \
0               8836                  47                 470   
1               8834                  57                 481   
2              11039                  76                 585   
3               1276                  10                  77   
4                854                   3                  48   

   pdf_files_accessed  docx_files_accessed  xlsx_files_accessed  exe_ratio  \
0                1878                    0                    0   0.005051   
1                1863                    0                    0   0.006114

In [14]:
df_email= pd.read_csv("/srv/cert/r4.2/email.csv")

In [ ]:
def extract_email_features_userlevel(df_email: pd.DataFrame) -> pd.DataFrame:
    df = df_email.copy()

    needed = {"user", "date"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"Faltam colunas obrigatórias no df_email: {missing}")

    # datetime
    df["datetime"] = pd.to_datetime(df["date"], errors="coerce", infer_datetime_format=True)
    df = df.dropna(subset=["datetime"])

    if "size" in df.columns:
        df["size_num"] = pd.to_numeric(df["size"], errors="coerce").fillna(0.0)
    else:
        df["size_num"] = 0.0

    if "content" in df.columns:
        df["text_len"] = df["content"].astype(str).fillna("").str.len()
    else:
        df["text_len"] = 0

    if "attachments" in df.columns:
        df["attachments_num"] = pd.to_numeric(df["attachments"], errors="coerce").fillna(0.0)
    else:
        df["attachments_num"] = 0.0

    g = df.groupby("user", observed=True)

    features = g.agg(
        n_email=("user", "size"),
        email_mean_size=("size_num", "mean"),
        email_mean_text_len=("text_len", "mean"),
        email_total_attachments=("attachments_num", "sum"),
    ).reset_index()

    if "pc" in df.columns:
        pc_feat = g["pc"].nunique().rename("email_unique_pcs_used").reset_index()
        features = features.merge(pc_feat, on="user", how="left")

    for col, out_name in [
        ("from", "email_unique_from"),
        ("to", "email_unique_to"),
        ("cc", "email_unique_cc"),
        ("bcc", "email_unique_bcc"),
    ]:
        if col in df.columns:
            tmp = g[col].nunique().rename(out_name).reset_index()
            features = features.merge(tmp, on="user", how="left")

    # limpa NaNs caso algum merge não encontre valores
    features = features.fillna(0)

    return features



In [17]:
email_user = extract_email_features_userlevel(df_email)
print(email_user.head())
print("shape:", email_user.shape)

      user  n_email  email_mean_size  email_mean_text_len  \
0  AAE0190     4711     30020.394184           346.391424   
1  AAF0535      480     30397.402083           344.593750   
2  AAF0791     3012     29958.497676           362.059761   
3  AAL0706      336     29828.181548           352.982143   
4  AAM0658      659     29895.532625           350.629742   

   email_total_attachments  email_unique_pcs_used  email_unique_from  \
0                     1780                      1                  2   
1                      364                      1                  2   
2                        0                      1                  3   
3                      145                      1                  2   
4                      613                      1                  2   

   email_unique_to  email_unique_cc  email_unique_bcc  
0             1529              364                 0  
1              299              103                 2  
2             1251              

In [19]:
df_http= pd.read_csv("/srv/cert/r4.2/http.csv")

In [ ]:
def compute_http_features_per_user(df_http: pd.DataFrame) -> pd.DataFrame:
    
    needed = {"user", "pc", "url"}
    missing = needed - set(df_http.columns)
    if missing:
        raise ValueError(f"Faltam colunas no df_http: {missing}")

    results = []

    for user, g in df_http.groupby("user", sort=False):
        total = 0
        unique_pcs = set()
        unique_urls = set()

        for row in g.itertuples(index=False):
            total += 1

            pc = getattr(row, "pc", None)
            if pc is not None:
                unique_pcs.add(pc)

            url = getattr(row, "url", None)
            if url is not None:
                unique_urls.add(url)

        results.append({
            "user": user,
            "total_http_events": total,
            "unique_pcs_used": len(unique_pcs),
            "unique_urls_visited": len(unique_urls),
        })

    return pd.DataFrame(results)


In [21]:
http_user = compute_http_features_per_user(df_http)
print(http_user.head())
print(http_user.shape)

      user  total_http_events  unique_pcs_used  unique_urls_visited
0  LRR0148              57200                1                  237
1  NGF0157              36450                1                  262
2  IRM0931              45315                1                  252
3  LAP0338              14123                1                  154
4  MOH0273              38095                1                  166
(1000, 4)


In [ ]:
from functools import reduce

def left_join_on_user(anchor: pd.DataFrame, others: list[pd.DataFrame]) -> pd.DataFrame:
    out = anchor.copy()
    for df in others:
        out = out.merge(df, on="user", how="left")
    return out

In [26]:
def prefix_features(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    df = df.copy()
    return df.rename(columns={c: f"{prefix}_{c}" for c in df.columns if c != "user"})

def left_join_on_user(anchor: pd.DataFrame, others: list[pd.DataFrame]) -> pd.DataFrame:
    out = anchor.copy()
    for df in others:
        out = out.merge(df, on="user", how="left")
    return out

logon_user_p  = prefix_features(logon_user,  "logon")
device_user_p = prefix_features(device_user, "device")
email_user_p  = prefix_features(email_user,  "email")
file_user_p   = prefix_features(file_user,   "file")
http_user_p   = prefix_features(http_user,   "http")

final_user_left = left_join_on_user(
    anchor=logon_user_p,
    others=[device_user_p, email_user_p, file_user_p, http_user_p]
)

print(final_user_left.shape)
print(final_user_left.head())


(1000, 49)
      user  logon_total_logons_day  logon_total_logoffs_day  \
0  AAE0190                     346                      346   
1  AAF0535                     164                      164   
2  AAF0791                     346                      346   
3  AAL0706                     346                      346   
4  AAM0658                     229                      229   

   logon_logon_to_logoff_ratio  logon_missing_logoff_count  \
0                          1.0                           0   
1                          1.0                           0   
2                          1.0                           0   
3                          1.0                           0   
4                          1.0                           0   

   logon_open_sessions_mean  logon_open_sessions_max  logon_distinct_pcs_used  \
0                       0.5                        1                      1.0   
1                       0.5                        1                      1

In [27]:
final_user_left.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 49 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   user                               1000 non-null   object 
 1   logon_total_logons_day             1000 non-null   int64  
 2   logon_total_logoffs_day            1000 non-null   int64  
 3   logon_logon_to_logoff_ratio        1000 non-null   float64
 4   logon_missing_logoff_count         1000 non-null   int64  
 5   logon_open_sessions_mean           1000 non-null   float64
 6   logon_open_sessions_max            1000 non-null   int64  
 7   logon_distinct_pcs_used            1000 non-null   float64
 8   logon_pc_switch_count              1000 non-null   int64  
 9   logon_sessions_total_duration      1000 non-null   float64
 10  device_unique_pcs_used             265 non-null    float64
 11  device_usb_connects                265 non-null    float6

In [28]:
final_user_left_clean = final_user_left.copy()

cols_to_fill = [c for c in final_user_left_clean.columns
                if c.startswith("device_") or c.startswith("file_")]

final_user_left_clean[cols_to_fill] = final_user_left_clean[cols_to_fill].fillna(0)

In [29]:
final_user_left_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 49 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   user                               1000 non-null   object 
 1   logon_total_logons_day             1000 non-null   int64  
 2   logon_total_logoffs_day            1000 non-null   int64  
 3   logon_logon_to_logoff_ratio        1000 non-null   float64
 4   logon_missing_logoff_count         1000 non-null   int64  
 5   logon_open_sessions_mean           1000 non-null   float64
 6   logon_open_sessions_max            1000 non-null   int64  
 7   logon_distinct_pcs_used            1000 non-null   float64
 8   logon_pc_switch_count              1000 non-null   int64  
 9   logon_sessions_total_duration      1000 non-null   float64
 10  device_unique_pcs_used             1000 non-null   float64
 11  device_usb_connects                1000 non-null   float6

In [ ]:
# Lista de usuários anômalos (label=1)
anomaly_users = [
    "RKD0604","RAB0589","MAR0955","JMB0308","EHD0584",
    "TAP0551","RGG0064","MAS0025","JRG0207","BDV0168",
    "WDD0366","KLH0596","FMG0527","BTL0226","BIH0745",
    "MCF0600","KPC0073","FTM0406","CAH0936","BLS0678",
    "MYD0978","LJR0523","GHL0460","DCH0843","AAM0658",
    "PPF0435","LQC0479","HJB0742","EHB0824","AJR0932",
    "XHW0498","TNM0961","VSS0154","RMW0542","RAR0725",
    "RHL0992","PSF0133","PNL0301","MOS0047","NWT0098",
    "MDH0580","KRL0501","LCC0819","JJM0203","IKR0401",
    "IUB0565","IJM0776","HBO0413","HXL0968","FSC0601",
    "EGD0132","EDB0714","DIB0285","DRR0162","CQW0652",
    "CCL0068","CEJ0109","ABC0174","AKR0057","AAF0535",
    "JTM0223","MPM0220","MSO0222","CCA0046","CSC0217",
    "GTD0219","JGT0221","JLM0364","BBS0039","BSS0369",
]

anomaly_set = set(anomaly_users)

# Criar label
final_user_left_clean["label"] = final_user_left_clean["user"].isin(anomaly_set).astype(int)

# Verificando
print("Total label=1:", final_user_left_clean["label"].sum())
print("Total label=0:", (final_user_left_clean["label"] == 0).sum())

# Mostrar os usuários marcados como 1
print(final_user_left_clean.loc[final_user_left_clean["label"] == 1, "user"].sort_values().tolist())


Total label=1: 70
Total label=0: 930
['AAF0535', 'AAM0658', 'ABC0174', 'AJR0932', 'AKR0057', 'BBS0039', 'BDV0168', 'BIH0745', 'BLS0678', 'BSS0369', 'BTL0226', 'CAH0936', 'CCA0046', 'CCL0068', 'CEJ0109', 'CQW0652', 'CSC0217', 'DCH0843', 'DIB0285', 'DRR0162', 'EDB0714', 'EGD0132', 'EHB0824', 'EHD0584', 'FMG0527', 'FSC0601', 'FTM0406', 'GHL0460', 'GTD0219', 'HBO0413', 'HJB0742', 'HXL0968', 'IJM0776', 'IKR0401', 'IUB0565', 'JGT0221', 'JJM0203', 'JLM0364', 'JMB0308', 'JRG0207', 'JTM0223', 'KLH0596', 'KPC0073', 'KRL0501', 'LCC0819', 'LJR0523', 'LQC0479', 'MAR0955', 'MAS0025', 'MCF0600', 'MDH0580', 'MOS0047', 'MPM0220', 'MSO0222', 'MYD0978', 'NWT0098', 'PNL0301', 'PPF0435', 'PSF0133', 'RAB0589', 'RAR0725', 'RGG0064', 'RHL0992', 'RKD0604', 'RMW0542', 'TAP0551', 'TNM0961', 'VSS0154', 'WDD0366', 'XHW0498']


In [31]:
users_in_df = set(final_user_left_clean["user"].unique())

missing = sorted(list(anomaly_set - users_in_df))
present = sorted(list(anomaly_set & users_in_df))

print("Usuários da lista presentes no dataset:", len(present))
print("Usuários da lista AUSENTES no dataset:", len(missing))


Usuários da lista presentes no dataset: 70
Usuários da lista AUSENTES no dataset: 0


In [32]:
final_user_left_clean

,user,logon_total_logons_day,logon_total_logoffs_day,logon_logon_to_logoff_ratio,logon_missing_logoff_count,logon_open_sessions_mean,logon_open_sessions_max,logon_distinct_pcs_used,logon_pc_switch_count,logon_sessions_total_duration,...,file_xlsx_files_accessed,file_exe_ratio,file_zip_ratio,file_pdf_ratio,file_docx_ratio,file_xlsx_ratio,http_total_http_events,http_unique_pcs_used,http_unique_urls_visited,label
0,AAE0190,346,346,1.0,0,0.5,1,1.000000,0,3478.333333,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,49478,1,230,0
1,AAF0535,164,164,1.0,0,0.5,1,1.000000,0,1318.866667,...,0.0,0.008403,0.033613,0.190476,0.0,0.0,4878,1,109,1
2,AAF0791,346,346,1.0,0,0.5,1,1.000000,0,3147.433333,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,32870,1,357,0
3,AAL0706,346,346,1.0,0,0.5,1,1.000000,0,3113.733333,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,3460,1,203,0
4,AAM0658,229,229,1.0,0,0.5,1,1.000000,0,2717.591389,...,0.0,0.000000,0.032258,0.193548,0.0,0.0,6498,1,127,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,ZKS0899,385,385,1.0,0,0.5,1,1.112717,78,3132.723333,...,0.0,0.005857,0.046269,0.199602,0.0,0.0,9976,1,142,0
996,ZMC0284,346,346,1.0,0,0.5,1,1.000000,0,3138.316667,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,32870,1,118,0
997,ZSB0649,346,346,1.0,0,0.5,1,1.000000,0,4170.950000,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,10034,1,112,0
998,ZSK0258,346,346,1.0,0,0.5,1,1.000000,0,3135.933333,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,32870,1,210,0
